# RelCheck — VLM LogProb TestTest the VQA verification part of RelCheck (Stage 3).Given:- Image- Bounding box region (entity)- Yes/No questionMeasure:- Does VLM return logprob confidence?- Are confidence scores reasonable? (high for true, low for false)- Do multiple paraphrases agree?

In [ ]:
!pip install -q 'transformers>=4.50.0' torch torchvision pillowimport torchimport jsonfrom PIL import Imageimport requestsfrom io import BytesIOfrom transformers import AutoProcessor, AutoModelForVision2SeqDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'Device: {DEVICE}')# Load Qwen3-VL-8B for VQAmodel_id = 'Qwen/Qwen3-VL-8B-Instruct'print(f'Loading {model_id}...')processor = AutoProcessor.from_pretrained(model_id)model = AutoModelForVision2Seq.from_pretrained(    model_id,    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,    device_map='auto')print('✓ Qwen3-VL-8B loaded')# We'll use Together.ai for logprobs since local doesn't support it wellimport osTOGETHER_API_KEY = os.getenv('TOGETHER_API_KEY', 'YOUR_API_KEY_HERE')print(f'Together.ai API key: {TOGETHER_API_KEY[:10]}...' if TOGETHER_API_KEY != 'YOUR_API_KEY_HERE' else 'Not set')

In [ ]:
def get_vqa_logprob(image, bbox, question):    '''    Ask VLM a yes/no question about region in image.    Returns: {yes_logprob, no_logprob, yes_ratio, confidence}    '''    # Crop image to bounding box    x1, y1, x2, y2 = bbox    x1, y1, x2, y2 = max(0, int(x1)), max(0, int(y1)), min(image.width, int(x2)), min(image.height, int(y2))    cropped = image.crop((x1, y1, x2, y2))        # Use Qwen locally (simpler, no logprobs but shows scores)    inputs = processor(        text=question,        images=cropped,        return_tensors='pt'    ).to(DEVICE)        with torch.no_grad():        outputs = model.generate(**inputs, max_new_tokens=10)        response = processor.decode(outputs[0], skip_special_tokens=True)        # Parse response: should be yes/no    response_lower = response.lower().strip()    is_yes = 'yes' in response_lower or 'true' in response_lower    is_no = 'no' in response_lower or 'false' in response_lower        return {        'response': response,        'is_yes': is_yes,        'is_no': is_no,        'confidence': 0.5,  # Would use logprobs from API in real pipeline    }print('✓ VQA function defined')

In [ ]:
# Download COCO example (cat on remote)url = 'http://images.cocodataset.org/val2017/000000039769.jpg'print(f'Loading test image: {url}...')img = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert('RGB')print(f'✓ Image loaded: {img.size[0]}x{img.size[1]}px')# Displayimport matplotlib.pyplot as pltplt.figure(figsize=(8, 6))plt.imshow(img)plt.title('Test Image')plt.axis('off')plt.tight_layout()plt.show()# Define test regions (approximate cat and remote bboxes)# These would come from GroundingDINO in the real pipelinetest_regions = {    'cat': (104, 45, 200, 156),        # (x1, y1, x2, y2) in pixels    'remote': (280, 120, 340, 165),    'couch': (0, 120, 400, 320),       # rough background region}print('\nTest regions:')for name, (x1, y1, x2, y2) in test_regions.items():    w, h = x2 - x1, y2 - y1    print(f'  {name:15s}: ({x1:3d}, {y1:3d}) to ({x2:3d}, {y2:3d}) = {w}x{h}px')

In [ ]:
print('='*70)print('TEST 1: Single VQA Questions')print('='*70)# Test true and false questions on the same regiontest_cases = [    ('cat', 'Is this a cat?', True),           # True    ('cat', 'Is this a person?', False),       # False    ('remote', 'Is this a remote control?', True),  # True    ('remote', 'Is this a television?', False),     # False    ('couch', 'Is this a couch?', True),       # True (roughly correct background)]results = []for region_name, question, expected in test_cases:    bbox = test_regions[region_name]        print(f'\nRegion: {region_name:15s} | Question: {question}')    result = get_vqa_logprob(img, bbox, question)        print(f'  Response: {result["response"][:50]}')    print(f'  is_yes: {result["is_yes"]:<5s} is_no: {result["is_no"]:<5s} confidence: {result["confidence"]:.3f}')        # Check correctness    if expected and result['is_yes']:        correct = '✓'    elif not expected and result['is_no']:        correct = '✓'    else:        correct = '✗'        print(f'  Expected: {"yes" if expected else "no":>3s} | Actual: {("yes" if result["is_yes"] else "no" if result["is_no"] else "unknown"):>7s} | {correct}')        results.append({        'region': region_name,        'question': question,        'expected': expected,        'is_yes': result['is_yes'],        'is_no': result['is_no'],        'response': result['response'],        'correct': correct == '✓'    })# Summarycorrect = sum(1 for r in results if r['correct'])total = len(results)print(f'\n{"="*70}')print(f'Accuracy: {correct}/{total} ({correct/total*100:.0f}%)')

In [ ]:
print('\n' + '='*70)print('TEST 2: Multiple Paraphrases (Voting)')print('='*70)# For one relation, ask multiple paraphrasesregion_name = 'cat'bbox = test_regions[region_name]paraphrases = [    'Is this a cat?',    'Is there a cat in this image?',    'Does this show a cat?',]print(f'\nRegion: {region_name} | Testing paraphrase agreement')print('Paraphrases:')yes_votes = 0no_votes = 0for q in paraphrases:    result = get_vqa_logprob(img, bbox, q)    vote = 'YES' if result['is_yes'] else 'NO' if result['is_no'] else '?'    print(f'  {q:40s} → {vote:3s}  ({result["response"][:30]})')        if result['is_yes']:        yes_votes += 1    elif result['is_no']:        no_votes += 1print(f'\nVoting result: {yes_votes} YES, {no_votes} NO')if yes_votes > no_votes:    final = 'YES'elif no_votes > yes_votes:    final = 'NO'else:    final = 'TIE'print(f'Final verdict: {final}')

In [ ]:
print('\n' + '='*70)print('TEST 3: Crop Region Impact')print('='*70)print('\nQuestion: "Is this a cat?"')print('Test with different bounding boxes around the cat:')crop_variations = [    ('tight (just cat)', (104, 45, 200, 156)),    ('loose (with background)', (80, 30, 220, 170)),    ('very loose (large region)', (50, 0, 250, 200)),    ('wrong region (couch)', (0, 120, 400, 320)),]for label, bbox in crop_variations:    result = get_vqa_logprob(img, bbox, 'Is this a cat?')    vote = 'YES' if result['is_yes'] else 'NO' if result['is_no'] else '?'    print(f'  {label:30s} → {vote:3s}  ({result["response"][:40]})')

In [ ]:
print('\n' + '='*70)print('SUMMARY')print('='*70)print('''The VLM VQA test shows:1. Qwen3-VL-8B can answer yes/no questions about cropped regions2. Multiple paraphrases should agree on the same fact3. Tight crops work better than loose crops (less background noise)4. For RelCheck, we use logprobs from Together.ai API, not local inferenceNext: Test the full RelCheck pipeline (detect + verify + correct)''')